# Baseline Training — All Models (Kaggle)
Trains each baseline model (SimpleCNN, ResNet-18/50, EfficientNet-B0, ViT-B/16, Swin-T, LightViT, DeiT-Small, FreqDetect, CLIP) using the **frozen split manifest** from HF.

**Prerequisite:** `train_mfft_base.ipynb` must have run at least once to bootstrap the manifest.

## Setup
1. **GPU**: T4 x2 or P100
2. **Attach all 11 datasets**
3. **Secret**: `HF_TOKEN`

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys, shutil
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "timm", "python-dotenv", "tqdm"], check=False)
print("Ready.")

In [ ]:
# Cell 2: Verify GPU & load manifest
import os, json, time
from pathlib import Path
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

# HF token
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
print(f"HF Token: {hf_token[:8]}...")

# Load manifest
from split_manifest_manager import SplitManifestManager
manifest_mgr = SplitManifestManager(hf_token=hf_token, run_id="baselines")
manifest, manifest_sha256 = manifest_mgr.download()
print(f"Manifest: v{manifest['version']} | {manifest['total_images']} images | sha256={manifest_sha256[:12]}...")

In [ ]:
# Cell 3: Create DataLoaders from manifest
from kaggle_dataset_loader import KaggleDatasetLoader

loader = KaggleDatasetLoader(
    manifest=manifest, manifest_sha256=manifest_sha256,
    input_root="/kaggle/input", image_size=224,
)
train_loader, val_loader, test_loader = loader.create_dataloaders(batch_size=64, num_workers=4)
print(f"Train: {len(loader.train_data)} | Val: {len(loader.val_data)} | Test: {len(loader.test_data)}")

In [ ]:
# Cell 4: Import baseline builders
sys.path.insert(0, str(REPO_DIR / "model"))
from src.baselines import (
    SimpleCNN, LightViT, count_parameters,
    resnet18, resnet50, efficientnet_b0, vit_b_16, swin_t,
)

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

CLASS_NAMES = ["real", "ai_generated", "deepfake"]
NUM_CLASSES = 3
EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 0.05

# Baseline registry
BASELINES = {
    "SimpleCNN":    lambda: SimpleCNN(num_classes=NUM_CLASSES),
    "LightViT":     lambda: LightViT(num_classes=NUM_CLASSES),
    "ResNet-18":    lambda: resnet18(num_classes=NUM_CLASSES),
    "ResNet-50":    lambda: resnet50(num_classes=NUM_CLASSES),
    "EfficientNet": lambda: efficientnet_b0(num_classes=NUM_CLASSES),
    "ViT-B/16":     lambda: vit_b_16(num_classes=NUM_CLASSES),
    "Swin-T":       lambda: swin_t(num_classes=NUM_CLASSES),
}
print(f"Baselines to train: {list(BASELINES.keys())}")

In [ ]:
# Cell 5: Training loop for one baseline
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def train_baseline(name, model_fn, train_loader, val_loader, test_loader, device, epochs=20):
    """Train a single baseline and return metrics."""
    model = model_fn().to(device)
    n_params = count_parameters(model)
    print(f"\n{'='*60}")
    print(f"Training {name} ({n_params:,} params)")
    print(f"{'='*60}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler(enabled=torch.cuda.is_available())

    best_f1 = 0.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f"  Train {name} {epoch:2d}/{epochs}", leave=False)
        for imgs, labels in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            t_loss += loss.item()
            t_correct += (logits.argmax(1) == labels).sum().item()
            t_total += labels.size(0)
            pbar.set_postfix(loss=f"{t_loss/t_total:.4f}", acc=f"{t_correct/t_total:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")
        scheduler.step()

        # Validate
        model.eval()
        v_loss, all_preds, all_labels, all_probs = 0, [], [], []
        with torch.no_grad():
            vbar = tqdm(val_loader, desc=f"  Val   {name} {epoch:2d}/{epochs}", leave=False)
            for imgs, labels in vbar:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                v_loss += criterion(logits, labels).item()
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(F.softmax(logits, dim=1).cpu().numpy())
                vbar.set_postfix(loss=f"{v_loss/(len(all_labels)):.4f}")

        y_true, y_pred, y_prob = np.array(all_labels), np.array(all_preds), np.array(all_probs)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        try:
            auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro") if y_prob.ndim == 2 else 0
        except Exception:
            auc = 0

        print(f"  Epoch {epoch:2d}/{epochs} | loss={t_loss/len(train_loader):.4f} acc={acc:.4f} f1={macro_f1:.4f} auc={auc:.4f}")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        history.append({"epoch": epoch, "loss": t_loss/len(train_loader), "acc": acc, "f1": macro_f1, "auc": auc})

    # Test with best model
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(F.softmax(logits, dim=1).cpu().numpy())

    y_true, y_pred, y_prob = np.array(all_labels), np.array(all_preds), np.array(all_probs)
    test_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    test_acc = accuracy_score(y_true, y_pred)
    try:
        test_auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        test_auc = 0

    print(f"  TEST: acc={test_acc:.4f} f1={test_f1:.4f} auc={test_auc:.4f}")

    return {
        "name": name, "n_params": n_params,
        "val_best_f1": best_f1, "test_acc": test_acc, "test_f1": test_f1, "test_auc": test_auc,
        "history": history,
    }

In [ ]:
# Cell 6: Train all baselines
results = []
for name, model_fn in BASELINES.items():
    try:
        r = train_baseline(name, model_fn, train_loader, val_loader, test_loader, device, epochs=EPOCHS)
        results.append(r)
    except Exception as e:
        print(f"  FAILED {name}: {e}")

# Save results
out_dir = Path("/kaggle/working/baseline_results")
out_dir.mkdir(exist_ok=True)
with open(out_dir / "baselines.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nResults saved to {out_dir / 'baselines.json'}")

In [ ]:
# Cell 7: Summary table
import pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in results])
df = df.sort_values("test_f1", ascending=False)
print("\n=== Baseline Results (sorted by test macro-F1) ===")
print(df.to_string(index=False))

# Upload to HF
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
api.upload_file(
    path_or_fileobj=str(out_dir / "baselines.json"),
    path_in_repo="results/baselines/baselines.json",
    repo_id="MohsinElis/mfft-checkpoints",
    repo_type="model",
)
print("\nUploaded to HF.")

In [ ]:
# Cell 8: Plot training curves and bar chart
import matplotlib.pyplot as plt

# Load results
with open(out_dir / "baselines.json") as f:
    results = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Bar chart: test F1 per baseline ---
names = [r["name"] for r in results]
test_f1s = [r["test_f1"] for r in results]
sorted_idx = np.argsort(test_f1s)[::-1]
names_sorted = [names[i] for i in sorted_idx]
f1s_sorted = [test_f1s[i] for i in sorted_idx]

ax = axes[0]
bars = ax.barh(names_sorted, f1s_sorted, color=plt.cm.viridis(np.linspace(0.3, 0.9, len(names_sorted))))
ax.set_xlabel("Test Macro-F1")
ax.set_title("Baseline Test F1")
ax.set_xlim(0, 1)
for bar, val in zip(bars, f1s_sorted):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2, f"{val:.4f}", va="center", fontsize=9)
ax.invert_yaxis()

# --- Loss curves ---
ax = axes[1]
for r in results:
    epochs_x = [h["epoch"] for h in r["history"]]
    losses = [h["loss"] for h in r["history"]]
    ax.plot(epochs_x, losses, marker=".", label=r["name"])
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.set_title("Training Loss Curves")
ax.legend(fontsize=7, loc="upper right")
ax.grid(True, alpha=0.3)

# --- F1 curves ---
ax = axes[2]
for r in results:
    epochs_x = [h["epoch"] for h in r["history"]]
    f1s = [h["f1"] for h in r["history"]]
    ax.plot(epochs_x, f1s, marker=".", label=r["name"])
ax.set_xlabel("Epoch")
ax.set_ylabel("Macro-F1")
ax.set_title("Validation F1 Curves")
ax.legend(fontsize=7, loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/baseline_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /kaggle/working/baseline_curves.png")